# 03. PyTorch Deep Neural Network for PMSM Fault Diagnosis

This notebook implements a PyTorch neural network pipeline for multi-class PMSM fault classification.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.data_loader import load_and_preprocess_data, PMSMDataset
from src.models import PyTorchANNModel

## 1. Data Ingestion & PyTorch DataLoaders

In [ ]:
data = load_and_preprocess_data(data_path=repo_root / 'data' / 'Classification.csv')
train_ds = PMSMDataset(data['X_train'], data['y_train'])
val_ds = PMSMDataset(data['X_val'], data['y_val'])
test_ds = PMSMDataset(data['X_test'], data['y_test'])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## 2. Initialize Model, Loss Function & Optimizer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PyTorchANNModel(input_size=5, num_classes=len(data['classes'])).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)

## 3. Training Loop

In [ ]:
epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out = model(x_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_b.size(0)
        total_correct += (out.argmax(dim=1) == y_b).sum().item()
        total_samples += x_b.size(0)
    
    if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
        acc = total_correct / total_samples
        print(f"Epoch [{epoch+1:02d}/{epochs}] Loss: {total_loss/total_samples:.4f} | Accuracy: {acc*100:.2f}%")

## 4. Test Evaluation

In [ ]:
model.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for x_b, y_b in test_loader:
        out = model(x_b.to(device))
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_trues.extend(y_b.numpy())

print(classification_report(all_trues, all_preds, target_names=[str(c) for c in data['classes']]))